# 03 — Classification du churn (J6–J7)

**Objectif :** prédire `Churn` (1 = le client part) avec 5 modèles, gérer le déséquilibre des classes, puis mesurer si la variable `cluster` (issue du clustering) améliore la prédiction.

**Règles respectées**
- `Churn` n'est jamais dans `X`.
- Imputation, scaling, encodage et SMOTE sont dans un `Pipeline` : ils sont ajustés **uniquement sur le train** (dans chaque pli de la validation croisée).
- Les modèles et stratégies sont **choisis par validation croisée stratifiée sur le train**. Le **test** ne sert qu'à l'évaluation finale.

**Plan :** données → baseline → 5 modèles × 3 stratégies (CV) → choix → évaluation sur le test → J7 : apport de `cluster` et du Feature Engineering → synthèse.

## 1. Imports et chargement

In [ ]:
import sys, os, warnings
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import RANDOM_STATE, TARGET
from src.models import (load_split, build_pipeline, build_baseline, MODEL_NAMES, STRATEGIES)
from src.evaluate import (stratified_cv, cv_scores, test_metrics, plot_confusions, plot_roc)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 200)

PROCESSED = "../data/processed"
for f in ["train_with_clusters.csv", "test_with_clusters.csv"]:
    assert os.path.exists(f"{PROCESSED}/{f}"), f"{f} manquant : lancer d'abord 02_clustering.ipynb (Run All)"

In [ ]:
X_train, X_test, y_train, y_test = load_split(PROCESSED)

assert TARGET not in X_train.columns and TARGET not in X_test.columns
print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print("Churn train : %.3f | test : %.3f" % (y_train.mean(), y_test.mean()))
print("Colonne cluster présente :", "cluster" in X_train.columns, "| valeurs :", sorted(X_train["cluster"].unique()))

Le split est celui de `src/preprocessing.py` (stratifié, `random_state` fixe) : mêmes clients que dans le notebook de clustering. `X` contient les variables préparées, les 3 variables de Feature Engineering et `cluster`. Chaque expérience choisit les colonnes dont elle a besoin via le `ColumnTransformer`.

## 2. Métriques et baseline

On regarde la classe **Churn** (la classe positive) :

| Métrique | Question |
|---|---|
| **Precision** | Parmi les clients prédits "churn", quelle part part vraiment ? |
| **Recall** | Parmi ceux qui partent vraiment, quelle part détecte-t-on ? |
| **F1** | Compromis entre Precision et Recall |
| **ROC-AUC** | Capacité à classer les clients par risque, indépendamment du seuil |

**Baseline :** un `DummyClassifier` qui prédit toujours "reste". C'est le plancher à dépasser.

In [ ]:
cv = stratified_cv(n_splits=5)                       # 5 plis stratifiés, sur le train
base = cv_scores(build_baseline(), X_train, y_train, cv)
pd.Series({k: round(v, 3) for k, v in base.items() if not k.endswith("_std")}, name="baseline (CV)").to_frame().T

**Lecture :** la baseline atteint ≈ 73,5 % d'**accuracy** sans rien apprendre, avec un Recall de 0 : elle ne détecte aucun client qui part. C'est pourquoi on juge les modèles sur Recall, F1 et ROC-AUC, pas sur l'accuracy.

## 3. Cinq modèles × trois stratégies contre le déséquilibre (validation croisée)

Pour chaque modèle, trois stratégies (≈ 26,5 % de churn) :
- **none** : aucun traitement ;
- **class_weight** : les erreurs sur la classe churn coûtent plus cher (`class_weight="balanced"`) ;
- **smote** : création de clients churn synthétiques **dans le Pipeline et sur le train du pli seulement**.

`Gradient Boosting` (scikit-learn) n'a pas de paramètre `class_weight` : cette combinaison n'est donc pas testée.
Les modèles gardent des réglages raisonnables par défaut ; l'optimisation est prévue en J8.

In [ ]:
rows = []
for strategy in STRATEGIES:
    for name in MODEL_NAMES:
        pipe = build_pipeline(name, strategy)
        if pipe is None:                      # combinaison inexistante
            continue
        res = cv_scores(pipe, X_train, y_train, cv)
        res.update(model=name, strategy=strategy)
        rows.append(res)
cv_table = pd.DataFrame(rows)

show = cv_table.set_index(["model", "strategy"])[["precision", "recall", "f1", "roc_auc", "f1_std", "roc_auc_std"]]
show.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=False)
for ax, metric, title in zip(axes, ["recall", "f1", "roc_auc"], ["Recall (churn)", "F1 (churn)", "ROC-AUC"]):
    sns.barplot(data=cv_table, x="model", y=metric, hue="strategy", ax=ax,
                palette={"none": "#9AA5B1", "class_weight": "#4C72B0", "smote": "#C44E52"})
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
    if metric == "roc_auc": ax.set_ylim(0.75, 0.87)
axes[0].legend(title="stratégie", fontsize=8); axes[1].get_legend().remove(); axes[2].get_legend().remove()
plt.tight_layout(); plt.show()

**Lecture (validation croisée sur le train) :**
- **Sans traitement**, les modèles détectent seulement 47 à 56 % des churners (Recall) avec une Precision de 62 à 68 %.
- **`class_weight` et SMOTE** font nettement monter le Recall (≈ 73 à 80 % pour Logistic Regression, Decision Tree et SVM) au prix d'une Precision plus faible (≈ 0,50 à 0,54 contre ≈ 0,67). Le F1 gagne 3 à 7 points (ex. Logistic Regression 0,59 → 0,63 ; SVM 0,55 → 0,62).
- **Le ROC-AUC ne bouge pratiquement pas** (Logistic Regression ≈ 0,845 dans les trois cas) : rééquilibrer change *où* on place le seuil de décision, pas la capacité à classer les clients.
- **Random Forest** profite peu de `class_weight` (F1 ≈ 0,55) et un peu de SMOTE (≈ 0,59) : il reste le moins bon en F1.
- **Gradient Boosting** passe de 0,58 à 0,62 de F1 avec SMOTE.
- Les meilleurs modèles (Logistic Regression, Gradient Boosting, SVM) sont à moins de 0,01 en F1, alors que l'écart-type entre plis est d'environ 0,02 : leurs différences ne sont **pas significatives**.

## 4. Choix de la stratégie par modèle

Pour chaque modèle, on garde la stratégie qui donne le meilleur **F1 en validation croisée** (pas sur le test).

In [ ]:
best = cv_table.loc[cv_table.groupby("model")["f1"].idxmax()].set_index("model")
best_strategy = best["strategy"].to_dict()
best[["strategy", "precision", "recall", "f1", "roc_auc"]].sort_values("f1", ascending=False).round(3)

## 5. Évaluation finale sur le test

Chaque modèle est ré-entraîné sur tout le train avec sa stratégie retenue, puis évalué **une seule fois** sur le test.

In [ ]:
fitted = {}
rows = {}
for name in MODEL_NAMES:
    pipe = build_pipeline(name, best_strategy[name]).fit(X_train, y_train)
    fitted[name] = pipe
    rows[f"{name} ({best_strategy[name]})"] = test_metrics(pipe, X_test, y_test)

dummy = build_baseline().fit(X_train, y_train)
rows["Baseline (toujours 'reste')"] = test_metrics(dummy, X_test, y_test)
test_table = pd.DataFrame(rows).T
test_table.round(3).sort_values("f1", ascending=False)

In [ ]:
plot_confusions(fitted, X_test, y_test, ncols=3)

In [ ]:
plot_roc(fitted, X_test, y_test)

**Lecture (test) :**
- Tous les modèles dépassent largement la baseline (Recall 0, ROC-AUC 0,5).
- **ROC-AUC** : Logistic Regression et Gradient Boosting arrivent en tête (≈ 0,84), Random Forest et SVM suivent (≈ 0,83).
- **Recall / F1** : les modèles rééquilibrés (Logistic Regression, Decision Tree, SVM) détectent environ 79 % des churners avec une Precision d'environ 0,51 à 0,53 : pour chaque client réellement perdu que l'on repère, on en signale à tort à peu près un autre. Gradient Boosting et Random Forest sont plus précis (≈ 0,57) mais détectent moins (65 % et 56 %).
- Les écarts de F1 entre les quatre meilleurs (≈ 0,61 à 0,63) sont petits et du même ordre que le bruit de la validation croisée. Le Decision Tree arrive en tête du F1 sur ce test mais pas en validation croisée, et son ROC-AUC est plus faible : on ne le retient pas sur ce seul résultat.
- Le **choix entre Recall et Precision est un choix métier** : si contacter un client coûte peu et perdre un client coûte cher, un Recall élevé est préférable.

## 6. J7 — La variable `cluster` et le Feature Engineering améliorent-ils la prédiction ?

On compare quatre jeux de variables, avec la **même stratégie (SMOTE)** pour tous les modèles afin que la comparaison soit équitable :

| Configuration | Variables |
|---|---|
| **A — initiales** | variables d'origine, sans Feature Engineering ni cluster |
| **B — + FE** | A + `n_services`, `avg_charge`, `tenure_group` |
| **C — initiales + cluster** | A + `cluster` (one-hot) |
| **D — FE + cluster** | B + `cluster` |

Comparaisons utiles : **A vs C** (apport du cluster sur les variables d'origine, demandé par le sujet), **A vs B** (apport du Feature Engineering), **B vs D** (cluster en plus du FE).
On utilise une validation croisée **5 plis × 2 répétitions** pour réduire le bruit (l'exécution prend environ 2 à 3 minutes).

In [ ]:
cv_rep = stratified_cv(n_splits=5, n_repeats=2)
configs = {"A initiales": (False, False), "B + FE": (False, True),
           "C initiales + cluster": (True, False), "D FE + cluster": (True, True)}

rows = []
for cfg_name, (use_cluster, use_fe) in configs.items():
    for name in MODEL_NAMES:
        pipe = build_pipeline(name, "smote", use_cluster=use_cluster, use_fe=use_fe)
        res = cv_scores(pipe, X_train, y_train, cv_rep)
        res.update(config=cfg_name, model=name)
        rows.append(res)
abl = pd.DataFrame(rows)

for metric in ["roc_auc", "f1", "recall"]:
    pv = abl.pivot(index="model", columns="config", values=metric)[list(configs)]
    pv.loc["MOYENNE"] = pv.mean()
    print(f"--- {metric} (validation croisée 5x2, SMOTE) ---")
    display(pv.round(3))
print("Écart-type moyen entre plis du ROC-AUC :", round(abl["roc_auc_std"].mean(), 3))

**Lecture :**
- Sur les quatre configurations, la moyenne du ROC-AUC reste autour de **0,834** et celle du F1 autour de **0,62** : les écarts sont d'environ **0,003 au maximum**, alors que l'écart-type entre plis est d'environ **0,012**.
- **Cluster (A vs C)** : aucune amélioration mesurable, ni pour un modèle linéaire, ni pour les arbres ou le boosting.
- **Feature Engineering (A vs B)** : pas d'amélioration non plus.
- **Conclusion honnête :** dans ce projet, ni `cluster` ni les trois variables créées n'améliorent la prédiction du churn.

**Pourquoi ?**
- `cluster` est calculé à partir des **mêmes variables** que celles que le modèle voit déjà : il en est un résumé, pas une information nouvelle. Un modèle supervisé retrouve ces combinaisons tout seul (surtout les arbres et le boosting).
- Les variables de Feature Engineering sont des **combinaisons** de variables existantes (nombre de services, charge moyenne, tranches d'ancienneté), donc largement redondantes.
- Le clustering a été fait **sans** `Churn` : il ne pouvait pas ajouter d'information sur la cible.

**Précaution sur la méthode :** K-Means a été ajusté sur tout le train, donc la validation croisée contient une légère fuite en faveur de `cluster`. Comme malgré cela aucun gain n'apparaît, la conclusion n'en est que plus solide.

**Ce que `cluster` apporte quand même :** une lecture métier. Les 4 segments (churn de 7 % à 57 %) servent à expliquer et à présenter les clients, notamment dans l'application Streamlit.

In [ ]:
# Vérification sur le test (évaluation, pas sélection) : effet du cluster, modèles en SMOTE
rows = []
for name in MODEL_NAMES:
    a = test_metrics(build_pipeline(name, "smote", False, False).fit(X_train, y_train), X_test, y_test)
    c = test_metrics(build_pipeline(name, "smote", True, False).fit(X_train, y_train), X_test, y_test)
    rows.append({"model": name, "roc_auc A": a["roc_auc"], "roc_auc C": c["roc_auc"],
                 "f1 A": a["f1"], "f1 C": c["f1"]})
pd.DataFrame(rows).set_index("model").round(3)

**Lecture :** sur le test, les écarts entre sans cluster (A) et avec cluster (C) restent de l'ordre de 0,01 au plus et ne vont pas tous dans le même sens : cela confirme l'absence d'effet net.

## 7. Synthèse et suite

**Résultats**
- **Baseline :** 73,5 % d'accuracy, Recall 0 : l'accuracy est trompeuse sur ce jeu déséquilibré.
- **Déséquilibre :** `class_weight` et SMOTE augmentent fortement le Recall (≈ 0,50 → 0,75–0,80 pour les modèles linéaires, SVM et arbre) et le F1 (+3 à +7 points) au prix de la Precision. Le ROC-AUC reste stable.
- **Modèles :** Logistic Regression et Gradient Boosting ont le meilleur ROC-AUC (≈ 0,84) ; Random Forest est le plus faible en F1 ; les différences entre les meilleurs sont du niveau du bruit.
- **Cluster et Feature Engineering :** pas d'amélioration mesurable de la prédiction.

**Candidats pour J8 (optimisation + MLflow)**
1. **Logistic Regression** : meilleur ROC-AUC, meilleur F1 en validation croisée, rapide, interprétable.
2. **Gradient Boosting** : ROC-AUC équivalent, Precision plus élevée que la Logistic Regression (≈ 0,59 contre ≈ 0,52), et plus de marge d'amélioration par réglage des hyperparamètres.

**À faire en J8 :** `GridSearchCV` / `RandomizedSearchCV` sur ces deux modèles (score : F1 ou ROC-AUC), comparaison avant / après, éventuel réglage du **seuil de décision** (0,5 par défaut) selon le compromis Recall / Precision souhaité, et suivi des expériences avec MLflow.